# Análise Exploratória — Preços de Combustíveis 2026

Este notebook usa exclusivamente as tabelas produzidas pelo pipeline do projeto a partir dos dados públicos da ANP.

**Objetivos:** entender cobertura, evolução semanal, diferenças geográficas e a relação etanol/gasolina sem digitar métricas manualmente.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'processed' / 'precos_semanais_2026.csv'
ANALYTICS = ROOT / 'data' / 'processed' / 'analytics'

if not DATA.exists():
    raise FileNotFoundError('Execute primeiro: python -m src.pipeline')

df = pd.read_csv(DATA, low_memory=False)
df['data_inicial'] = pd.to_datetime(df['data_inicial'], errors='coerce')
df.head()

## 1. Cobertura dos dados

In [ ]:
resumo = pd.DataFrame({
    'indicador': ['linhas', 'produtos', 'semanas', 'níveis geográficos'],
    'valor': [
        len(df),
        df['produto'].nunique(),
        df['data_inicial'].nunique(),
        df['nivel_geografico'].nunique(),
    ],
})
resumo

In [ ]:
pd.crosstab(df['nivel_geografico'], df['produto'])

## 2. Evolução semanal no nível Brasil

In [ ]:
brasil = df[df['nivel_geografico'].str.lower().eq('brasil')].copy()
brasil = brasil.sort_values('data_inicial')

fig, ax = plt.subplots(figsize=(12, 6))
for produto, grupo in brasil.groupby('produto'):
    ax.plot(grupo['data_inicial'], grupo['preco_medio_revenda'], label=produto)

ax.set_title('Evolução semanal do preço médio de revenda — Brasil')
ax.set_xlabel('Semana')
ax.set_ylabel('Preço médio de revenda')
ax.legend()
fig.autofmt_xdate()
plt.show()

## 3. Ranking das UFs na última semana

In [ ]:
ranking_uf_path = ANALYTICS / 'ranking_ufs_ultima_semana.csv'
ranking_uf = pd.read_csv(ranking_uf_path)

produtos = ranking_uf['produto'].dropna().unique().tolist()
produto_exemplo = next(
    (p for p in produtos if 'GASOLINA' in p.upper() and 'ADITIV' not in p.upper()),
    produtos[0],
)

top = (
    ranking_uf[ranking_uf['produto'].eq(produto_exemplo)]
    .sort_values('preco_medio_revenda', ascending=False)
    .head(10)
    .sort_values('preco_medio_revenda')
)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top['uf'], top['preco_medio_revenda'])
ax.set_title(f'10 maiores preços médios por UF — {produto_exemplo}')
ax.set_xlabel('Preço médio de revenda')
ax.set_ylabel('UF')
plt.show()

## 4. Relação etanol/gasolina por município

In [ ]:
ratio_path = ANALYTICS / 'etanol_gasolina_ultima_semana.csv'
ratio = pd.read_csv(ratio_path)

ratio['relacao_etanol_gasolina_pct'].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(ratio['preco_gasolina_comum'], ratio['preco_etanol'])
ax.set_title('Preço do etanol × gasolina comum — última semana')
ax.set_xlabel('Preço médio da gasolina comum')
ax.set_ylabel('Preço médio do etanol')
plt.show()

## 5. Cuidados de interpretação

- Os níveis Brasil, região, estado e município têm grãos diferentes e não devem ser misturados sem filtro.
- A média mensal produzida pelo projeto é derivada das observações semanais; não substitui a série mensal oficial da ANP.
- Valores extremos são investigados, não removidos automaticamente.
- A razão etanol/gasolina é apresentada como indicador observado. Qualquer limiar econômico deve ser documentado separadamente.